# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**

* **Nombres y matrículas:**

*    Lucía Guadalupe Macías Morales   A01796976
*    Luz Copelia Minutti Pérez        A01796921
*    Ángel Irwin Briseño Fierro       A01796844
*    Antonio Molina Suárez            A01796889

* **Número de Equipo:**


* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [5]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
!pip install openai-whisper
import time
import os
import requests
import zipfile
import re
import json
import pandas as pd
import numpy as np
import whisper
# Para el modelo LDA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Para el LLM
import google.generativeai as genai

print("✅ Librerías instaladas e importadas correctamente.")

✅ Librerías instaladas e importadas correctamente.


In [6]:
from google.colab import drive
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# URLs y configuración
url_zip = "https://www.gutenberg.org/files/21144/21144-mp3.zip"
ruta_zip = "esopo_audios.zip"
carpeta_destino = base_path+"audios_esopo"

# IDs de las fábulas que necesitas
ids_requeridos = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]

# --- Descarga del archivo zip ---
print(f"Descargando {url_zip} ...")
respuesta = requests.get(url_zip, stream=True)

if respuesta.status_code == 200:
    with open(ruta_zip, 'wb') as archivo_zip:
        for chunk in respuesta.iter_content(chunk_size=128):
            archivo_zip.write(chunk)
    print("✅ Descarga completada.")
else:
    print(f"❌ Error: No se pudo descargar el archivo. Código: {respuesta.status_code}")
    # En caso de falla, podemos probar con un mirror alternativo
    url_mirror = "http://mirror.eu.ossplanet.net/gutenberg/2/1/1/4/21144/21144-mp3.zip"
    print(f"Intentando con mirror: {url_mirror}")
    respuesta = requests.get(url_mirror, stream=True)
    if respuesta.status_code == 200:
        with open(ruta_zip, 'wb') as archivo_zip:
            for chunk in respuesta.iter_content(chunk_size=128):
                archivo_zip.write(chunk)
        print("✅ Descarga completada desde el mirror.")
    else:
        print("❌ Error también en el mirror. Verificar la conexión.")

# --- Extracción de los archivos necesarios ---
print("\nExtrayendo archivos...")
os.makedirs(carpeta_destino, exist_ok=True)

archivos_extraidos = []

with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
    # Listar todos los archivos dentro del ZIP (incluyendo subcarpetas)
    for miembro in zip_ref.namelist():
        # Busca archivos .mp3 que contengan el patrón "21144-XX.mp3" en cualquier subcarpeta
        if miembro.endswith('.mp3') and '21144-' in miembro:
            # Obtener solo el nombre del archivo (sin la ruta de subcarpetas)
            nombre_archivo = os.path.basename(miembro)

            # Extrae el número de fábula del nombre del archivo
            # El nombre tiene formato: 21144-XX.mp3
            num_str = nombre_archivo.replace('.mp3', '').replace('21144-', '')

            # Convertimos a entero para comparar (maneja '01' -> 1)
            try:
                num_fabula = int(num_str)
                if num_fabula in ids_requeridos:
                    # Extrae el archivo (manteniendo el nombre original)
                    with zip_ref.open(miembro) as fuente, open(os.path.join(carpeta_destino, nombre_archivo), 'wb') as destino:
                        destino.write(fuente.read())
                    archivos_extraidos.append(nombre_archivo)
                    print(f"  - Extraído: {nombre_archivo} (Fábula {num_fabula}) desde {miembro}")
            except ValueError:
                # Si no se puede convertir a entero, ignorar
                continue

print(f"\n✅ Extracción completada. Se extrajeron {len(archivos_extraidos)} archivos en la carpeta: '{carpeta_destino}'")

# Verificar que se hayan extraído los 10 archivos
if len(archivos_extraidos) == 10:
    print("🎉 ¡Éxito! Se extrajeron las 10 fábulas requeridas.")
else:
    print(f"⚠️ Advertencia: Solo se extrajeron {len(archivos_extraidos)} de 10 archivos. Revisa los nombres.")
    print("Archivos esperados:", [f"21144-{str(i).zfill(2)}.mp3" for i in ids_requeridos])
    print("\n📂 Contenido completo del ZIP (primeros 20 archivos):")
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        for i, nombre in enumerate(zip_ref.namelist()[:20]):
            print(f"    {nombre}")

Descargando https://www.gutenberg.org/files/21144/21144-mp3.zip ...
✅ Descarga completada.

Extrayendo archivos...
  - Extraído: 21144-01.mp3 (Fábula 1) desde mp3/21144-01.mp3
  - Extraído: 21144-04.mp3 (Fábula 4) desde mp3/21144-04.mp3
  - Extraído: 21144-05.mp3 (Fábula 5) desde mp3/21144-05.mp3
  - Extraído: 21144-06.mp3 (Fábula 6) desde mp3/21144-06.mp3
  - Extraído: 21144-14.mp3 (Fábula 14) desde mp3/21144-14.mp3
  - Extraído: 21144-22.mp3 (Fábula 22) desde mp3/21144-22.mp3
  - Extraído: 21144-24.mp3 (Fábula 24) desde mp3/21144-24.mp3
  - Extraído: 21144-25.mp3 (Fábula 25) desde mp3/21144-25.mp3
  - Extraído: 21144-26.mp3 (Fábula 26) desde mp3/21144-26.mp3
  - Extraído: 21144-27.mp3 (Fábula 27) desde mp3/21144-27.mp3

✅ Extracción completada. Se extrajeron 10 archivos en la carpeta: '/content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/audios_esopo'
🎉 ¡Éxito! Se extrajeron las 10 fábulas requeridas.


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

In [10]:
# Cargar el modelo Whisper
print("Cargando el modelo Whisper...")
modelo_whisper = whisper.load_model("small")  # small es buen balance velocidad/precisión
print("Modelo Whisper cargado.")
prompt_ayuda = "Las fábulas de Esopo. Había una vez un lobo, un león y una zorra. Fin de la fábula. Esta grabación es de dominio público."
# Crear carpeta para guardar las transcripciones
carpeta_transcripciones = base_path+"transcripciones"
os.makedirs(carpeta_transcripciones, exist_ok=True)

# Ruta donde se guardaron los audios
ruta_audios = base_path+"audios_esopo"

print("\n--- Iniciando Transcripciones (esto puede tomar varios minutos) ---")
print(" Aproximadamente 10-20 segundos por cada audio...\n")

transcripciones = {}

for i, nombre_archivo in enumerate(os.listdir(ruta_audios), 1):
    if nombre_archivo.endswith(".mp3"):
        ruta_completa = os.path.join(ruta_audios, nombre_archivo)

        # Nombre del archivo de texto donde guardaremos la transcripción
        nombre_txt = nombre_archivo.replace('.mp3', '_transcripcion.txt')
        ruta_txt = os.path.join(carpeta_transcripciones, nombre_txt)

        print(f"[{i}/10] Transcribiendo: {nombre_archivo}")

        # Transcribir usando Whisper
        resultado = modelo_whisper.transcribe(ruta_completa, language="es",
                                              initial_prompt=prompt_ayuda,
                                              temperature=0.0)
        texto_transcrito = resultado["text"]

        # Guardar en archivo de texto individual
        with open(ruta_txt, 'w', encoding='utf-8') as f:
            f.write(texto_transcrito)

        print(f"Guardado en: {ruta_txt}\n")
        time.sleep(0.5)

print("=" * 50)
print("¡Transcripción completada!")
print(f"📁 Las transcripciones se guardaron en la carpeta: '{carpeta_transcripciones}'")
print("=" * 50)

Cargando el modelo Whisper...


100%|███████████████████████████████████████| 461M/461M [00:08<00:00, 55.3MiB/s]


Modelo Whisper cargado.

--- Iniciando Transcripciones (esto puede tomar varios minutos) ---
 Aproximadamente 10-20 segundos por cada audio...

[1/10] Transcribiendo: 21144-14.mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/transcripciones/21144-14_transcripcion.txt

[2/10] Transcribiendo: 21144-04.mp3


...

Guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/transcripciones/21144-22_transcripcion.txt

[9/10] Transcribiendo: 21144-01.mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/transcripciones/21144-01_transcripcion.txt

[10/10] Transcribiendo: 21144-24.mp3


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Guardado en: /content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/transcripciones/21144-24_transcripcion.txt

¡Transcripción completada!
📁 Las transcripciones se guardaron en la carpeta: '/content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/transcripciones'


# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [22]:
import re
import json
import os

def eliminar_inicio_final_comun(texto):
    """
    Elimina inicio y final común de las fábulas.
    """
    # 0. Limpiar espacios iniciales para normalizar antes de aplicar patrones
    texto = re.sub(r'\s+', ' ', texto).strip()

    # 1. Quitar INICIO: tomar todo DESPUÉS de "fábula número X" o "Fábula nº X"
    match = re.search(r'(?:fábula\s+número|fábula\s+nº)\s+\d+\s*[.,:;]?\s*(.*)', texto, re.IGNORECASE)
    if match:
        texto = match.group(1)
        # Re-normalize whitespace after potentially cutting the beginning
        texto = re.sub(r'\s+', ' ', texto).strip()

    # 2. Quitar créditos de sitios web (e.g., 'www.esopo.org')
    patron_web_credit = r'www\.[a-zA-Z0-9-]+\.org\s*$'
    texto = re.sub(patron_web_credit, "", texto, flags=re.IGNORECASE)

    # 3. Quitar créditos de subtítulos (e.g., 'Subtítulos por la comunidad de Amara.org')
    patron_subtitulos = r'Subtítulos\s+por\s+la\s+comunidad\s+de\s+Amara\.org\s*$' # This was step 2 before
    texto = re.sub(patron_subtitulos, "", texto, flags=re.IGNORECASE)

    # 4. Quitar CUALQUIER texto de crédito de LibriVox.org que pueda quedar al final
    # Este patrón cubre 'Las fábulas de Esopo. Grabado para LibriVox.org por [Name/Phrase].'
    patron_credito_librivox = r'Las\s+f[aá]bulas\s+de\s+Esopo\.\s+Grabado\s+para\s+LibriVox\.org\s+por\s+.*$'
    texto = re.sub(patron_credito_librivox, "", texto, flags=re.IGNORECASE)

    # 5. Quitar FINAL: eliminar la frase explícita de "Fin de la fábula... dominio público."
    # Se añade '$' al final para asegurar que solo se elimine si está al final del texto.
    # Se ha ajustado el patrón para hacer el segundo 'es' opcional.
    patron_fin_dominio = r'F[ií]n\s*de(?:\s+la)?\s+f[aá]b[ou]la\s*[.,;:]?\s*[Ee]sta\s+(?:es\s+una\s+)?gra[bv]aci[oó]n\s+(?:es\s+)?(?:de[l]?\s+)?dominio\s+p[uú]blico\.?\s*$'
    texto = re.sub(patron_fin_dominio, "", texto, flags=re.IGNORECASE)

    # 6. Limpiar espacios (de nuevo, ya que las sustituciones pueden dejar espacios extra)
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto


# --- Cargar transcripciones ---
carpeta_transcripciones = base_path+"transcripciones"
transcripciones = {}

print("--- Cargando transcripciones ---")
for nombre_txt in os.listdir(carpeta_transcripciones):
    if nombre_txt.endswith('_transcripcion.txt'):
        ruta_txt = os.path.join(carpeta_transcripciones, nombre_txt)
        with open(ruta_txt, 'r', encoding='utf-8') as f:
            contenido = f.read()
        nombre_audio = nombre_txt.replace('_transcripcion.txt', '.mp3')
        transcripciones[nombre_audio] = contenido
        print(f"   Cargado: {nombre_audio}")

print(f"\n Cargadas {len(transcripciones)} transcripciones.\n")

# --- Aplicar limpieza ---
print("--- Eliminando inicio y final común ---")
fabulas_limpias = {}

for nombre, texto in transcripciones.items():
    texto_limpio = eliminar_inicio_final_comun(texto)
    fabulas_limpias[nombre] = texto_limpio
    print(f"   {nombre}")

# --- Guardar ---
with open(base_path+"fabulas_sin_intro_outro.json", "w", encoding="utf-8") as f:
    json.dump(fabulas_limpias, f, ensure_ascii=False, indent=4)

print("\n Guardado: fabulas_sin_intro_outro.json")

# --- Verificación ---
print("\n" + "=" * 60)
print("VERIFICACIÓN")
print("=" * 60)

for nombre, texto in transcripciones.items():
    # Checking for specific phrases to confirm removal
    # We'll check the cleaned text directly to see if the ending is gone.
    if re.search(r'F[ií]n\s*de(?:\s+la)?\s+f[aá]b[ou]la', fabulas_limpias[nombre], re.IGNORECASE) or \
       re.search(r'Las\s+f[aá]bulas\s+de\s+Esopo\.\s+Grabado\s+para\s+LibriVox\.org', fabulas_limpias[nombre], re.IGNORECASE) or \
       re.search(r'Subtítulos\s+por\s+la\s+comunidad\s+de\s+Amara\.org', fabulas_limpias[nombre], re.IGNORECASE) or \
       re.search(r'www\.[a-zA-Z0-9-]+\.org', fabulas_limpias[nombre], re.IGNORECASE):
        print(f"\n Error en limpieza para: {nombre}")
        print("ORIGINAL (final):")
        lineas = texto.split('\n')
        for linea in lineas[-3:]:
            print(f"  {linea}")
        print("\nLIMPIO (final - AUN CON RESIDUOS):")
        print(f"  {fabulas_limpias[nombre][-150:]}")
        break
    else:
        print(f"\n {nombre}")
        print("ORIGINAL (final):")
        lineas = texto.split('\n')
        for linea in lineas[-3:]:
            print(f"  {linea}")
        print("\nLIMPIO (final):")
        print(f"  {fabulas_limpias[nombre][-150:]}")

--- Cargando transcripciones ---
   Cargado: 21144-14.mp3
   Cargado: 21144-04.mp3
   Cargado: 21144-05.mp3
   Cargado: 21144-26.mp3
   Cargado: 21144-06.mp3
   Cargado: 21144-25.mp3
   Cargado: 21144-27.mp3
   Cargado: 21144-22.mp3
   Cargado: 21144-01.mp3
   Cargado: 21144-24.mp3

 Cargadas 10 transcripciones.

--- Eliminando inicio y final común ---
   21144-14.mp3
   21144-04.mp3
   21144-05.mp3
   21144-26.mp3
   21144-06.mp3
   21144-25.mp3
   21144-27.mp3
   21144-22.mp3
   21144-01.mp3
   21144-24.mp3

 Guardado: fabulas_sin_intro_outro.json

VERIFICACIÓN

 21144-14.mp3
ORIGINAL (final):
   Las fábulas de Esopo. Grabado para LibriVox.org por el ochito. Fábula número 74. El lobo y el cabrito encerrado. Protegido por la seguridad del corral de una casa, un cabrito vio pasar a un lobo y comenzó a insultarle burlándose ampliamente de él. El lobo serenamente le replicó. Infeliz. Sé que no eres tú quien me está insultando, sino el sitio en que te encuentras. Muy a menudo no es el val

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

In [23]:

# ============================================
# JUSTIFICACIÓN DE NUESTRA DECISIÓN
# ============================================
#
# DECISIÓN: No aplicaremos limpieza agresiva
#
# RAZONES:
# 1. Los textos de las fábulas son MUY CORTOS (aproximadamente 150-300 palabras)
# 2. Eliminar stopwords (artículos, preposiciones, conjunciones) eliminaría
#    información crucial para entender la narrativa.
# 3. Las palabras cortas como "se", "le", "lo", "ya" son importantes en español.
# 4. El LDA funciona mejor con textos cortos si se mantiene el vocabulario completo.
# 5. Las transcripciones de Whisper ya vienen con puntuación estándar.
#
# LIMPIEZA MÍNIMA QUE SÍ APLICAMOS:
# - Convertir a minúsculas para uniformidad
# - Eliminar caracteres no alfabéticos (números, símbolos extraños)
# - Eliminar espacios extras
# - Stopwords
# NO ELIMINAMOS:
# - Palabras cortas
# - Palabras con poca frecuencia
# - Puntuación básica (.,;:?!)
#
# ============================================

# Descargar stopwords en español
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stopwords_es = set(stopwords.words('spanish'))

# Agregar más palabras comunes que no aportan significado
stopwords_adicionales = {
    'si', 'no', 'ya', 'más', 'muy', 'tan', 'como', 'cuando', 'donde',
    'porque', 'entonces', 'ahora', 'también', 'sino', 'tras', 'ante',
    'bajo', 'contra', 'desde', 'durante', 'entre', 'hacia', 'hasta',
    'mediante', 'para', 'según', 'sin', 'sobre', 'vs', 'éste', 'ésta',
    'aquél', 'aquella', 'eso', 'esa', 'ese', 'esto', 'esta', 'este'
}
stopwords_es.update(stopwords_adicionales)

print(f"📚 Total de stopwords a eliminar: {len(stopwords_es)}")
print(f"Ejemplos: {list(stopwords_es)[:15]}\n")

def limpieza_minima(texto):
    """
    Aplica SOLO limpieza mínima:
    - Convertir a minúsculas
    - Eliminar caracteres no alfabéticos (excepto espacios y puntuación básica)
    - Eliminar espacios extras
    """
    # Convertir a minúsculas
    texto = texto.lower()

    # Eliminar caracteres no deseados (números, símbolos raros)
    # Pero mantener letras, espacios y puntuación básica
    texto = re.sub(r'[^a-záéíóúñü\s\.\,\;\:\!\?\-]', '', texto)

    # Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto)

    # Eliminar espacios al inicio y final
    texto = texto.strip()

    #  Eliminar stopwords
    palabras = texto.split()
    palabras_filtradas = [p for p in palabras if p not in stopwords_es]
    texto = ' '.join(palabras_filtradas)

    return texto


# --- Cargar los textos del Ejercicio 2b desde el JSON --

json_path = base_path+"fabulas_sin_intro_outro.json"

if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        fabulas_procesadas = json.load(f)
    print(f" Cargadas {len(fabulas_procesadas)} fábulas desde {json_path}")
else:
    print("No se encuentra el archivo. Asegúrate de haber ejecutado el Ejercicio 2b primero.")
    fabulas_procesadas = {}

# --- Aplicar limpieza mínima ---
print("\n--- Aplicando limpieza MÍNIMA (sin eliminar stopwords) ---")

fabulas_limpias = {}

for nombre_archivo, texto in fabulas_procesadas.items():
    texto_limpio = limpieza_minima(texto)
    fabulas_limpias[nombre_archivo] = texto_limpio
    print(f" Limpiada: {nombre_archivo}")

# --- Guardar resultados del Ejercicio 3 ---
with open(base_path+"fabulas_limpias.json", "w", encoding="utf-8") as f:
    json.dump(fabulas_limpias, f, ensure_ascii=False, indent=4)
print("\nGuardado: fabulas_limpias.json")

# --- Mostrar comparación ---
print("\n" + "=" * 70)
print(" COMPARACIÓN DEL PROCESO")
print("=" * 70)

primer_audio = list(fabulas_procesadas.keys())[0]
print(f"\n {primer_audio}")
print("\n DESPUÉS DEL EJERCICIO 2b (sin intro/outro):")
print(fabulas_procesadas[primer_audio])
print("\n DESPUÉS DEL EJERCICIO 3 (limpieza mínima):")
print(fabulas_limpias[primer_audio])

# Mostrar estadísticas de longitud
print("\n" + "=" * 70)
print(" ESTADÍSTICAS DE LONGITUD")
print("=" * 70)
print(f"{'Archivo':<25} {'Ej2b (chars)':<15} {'Ej3 (chars)':<15} {'Diferencia':<10}")
print("-" * 70)
for nombre in list(fabulas_limpias.keys()):
    len_2b = len(fabulas_procesadas[nombre])
    len_3 = len(fabulas_limpias[nombre])
    diff = len_2b - len_3
    print(f"{nombre:<25} {len_2b:<15} {len_3:<15} {diff:<10}")



📚 Total de stopwords a eliminar: 328
Ejemplos: ['suyo', 'estuviéramos', 'habríais', 'tengan', 'ellas', 'habrás', 'nuestras', 'tienen', 'unos', 'hubieseis', 'e', 'sentida', 'estuviera', 'estarán', 'tenéis']

 Cargadas 10 fábulas desde /content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/fabulas_sin_intro_outro.json

--- Aplicando limpieza MÍNIMA (sin eliminar stopwords) ---
 Limpiada: 21144-14.mp3
 Limpiada: 21144-04.mp3
 Limpiada: 21144-05.mp3
 Limpiada: 21144-26.mp3
 Limpiada: 21144-06.mp3
 Limpiada: 21144-25.mp3
 Limpiada: 21144-27.mp3
 Limpiada: 21144-22.mp3
 Limpiada: 21144-01.mp3
 Limpiada: 21144-24.mp3

Guardado: fabulas_limpias.json

 COMPARACIÓN DEL PROCESO

 21144-14.mp3

 DESPUÉS DEL EJERCICIO 2b (sin intro/outro):
El lobo y el cabrito encerrado. Protegido por la seguridad del corral de una casa, un cabrito vio pasar a un lobo y comenzó a insultarle burlándose ampliamente de él. El lobo serenamente le replicó. Infeliz. Sé que no eres tú quien me est

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# **Ejercicio 4:**

In [24]:
# @title **Ejercicio 4 - Extracción de palabras clave con LDA**

import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# --- Cargar textos limpios ---
print("--- Cargando textos limpios ---")
with open(base_path+"fabulas_limpias.json", "r", encoding="utf-8") as f:
    fabulas = json.load(f)

print(f" Cargadas {len(fabulas)} fábulas\n")

# --- Preparar datos ---
nombres = list(fabulas.keys())
textos = list(fabulas.values())

# --- Vectorizar ---
vectorizador = CountVectorizer(max_features=100, stop_words=None)
matriz_dtm = vectorizador.fit_transform(textos)
nombres_palabras = vectorizador.get_feature_names_out()

print(f" Vocabulario: {len(nombres_palabras)} palabras únicas\n")

# --- Función para extraer palabras clave de una fábula ---
def obtener_palabras_clave(texto, n_palabras=20):
    """Extrae palabras clave de un solo texto usando LDA con 1 tópico."""
    vectorizador_local = CountVectorizer()
    matriz_local = vectorizador_local.fit_transform([texto])
    lda = LatentDirichletAllocation(n_components=1, random_state=42)
    lda.fit(matriz_local)

    palabras = vectorizador_local.get_feature_names_out()
    componentes = lda.components_[0]
    indices = componentes.argsort()[:-n_palabras-1:-1]

    return [palabras[i] for i in indices]

# --- Extraer palabras clave para cada fábula ---
print("--- Extrayendo palabras clave (20 por fábula) ---")
palabras_clave_por_fabula = {}

for i, (nombre, texto) in enumerate(fabulas.items(), 1):
    palabras = obtener_palabras_clave(texto, n_palabras=20)
    palabras_clave_por_fabula[nombre] = palabras
    print(f"  {i}. {nombre}: {', '.join(palabras[:8])}...")

# --- Guardar resultados ---
with open(base_path+"palabras_clave_lda.json", "w", encoding="utf-8") as f:
    json.dump(palabras_clave_por_fabula, f, ensure_ascii=False, indent=4)

print("\n Guardado: palabras_clave_lda.json")

# --- Mostrar todas las palabras clave ---
print("\n" + "=" * 60)
print("TODAS LAS PALABRAS CLAVE POR FÁBULA")
print("=" * 60)

for nombre, palabras in palabras_clave_por_fabula.items():
    print(f"\n {nombre}")
    print(f"   {', '.join(palabras)}")

--- Cargando textos limpios ---
 Cargadas 10 fábulas

 Vocabulario: 100 palabras únicas

--- Extrayendo palabras clave (20 por fábula) ---
  1. 21144-14.mp3: lobo, cabrito, valor, vio, él, serenamente, seguridad, replicó...
  2. 21144-04.mp3: lobo, hueso, paga, pidió, garganta, grulla, cabeza, boca...
  3. 21144-05.mp3: caballo, cebada, lobo, sembrado, ruido, repuso, rato, preferido...
  4. 21144-26.mp3: campanilla, perro, virtudes, sonando, solo, tanto, sé, razón...
  5. 21144-06.mp3: lobo, asno, ley, vez, todos, sé, secretado, repartiese...
  6. 21144-25.mp3: carnicero, perro, viéndole, trozo, suceda, volvió, pensar, penetró...
  7. 21144-27.mp3: perro, león, volvió, soporta, siquiera, siempre, zorra, rápidamente...
  8. 21144-22.mp3: luego, perro, almeja, huevos, tragó, tomes, ver, veo...
  9. 21144-01.mp3: lobo, templo, cordero, ser, dios, víctima, vamos, tener...
  10. 21144-24.mp3: reflejo, río, perro, ajeno, propio, carne, pedazo, trozo...

 Guardado: palabras_clave_lda.json

TO

# **Ejercicio 5a y 5b:**

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

In [27]:
import json
from google.colab import userdata
# It appears there might be an environment issue causing 'google.genai' to be loaded instead of 'google.generativeai'.
# Please ensure `google-generativeai` is installed and consider restarting the Colab runtime if errors persist.
import google.generativeai as genai
import time # Import time for adding pauses

# --- Configurar Gemini ---
try:
    API_KEY = userdata.get('GEMINI_API_KEY2')
    print("✅ API Key obtenida")
except Exception as e:
    print(f"❌ Error al obtener la API Key de secrets: {e}")
    print("Por favor, asegúrate de haber guardado tu GEMINI_API_KEY en Colab Secrets.")
    # Fallback to input if secrets fail, though the primary request is to use secrets
    API_KEY = input("🔑 API Key (Alternativa): ")

genai.configure(api_key=API_KEY)
modelo = genai.GenerativeModel('gemini-2.5-flash')

# --- Cargar palabras clave ---
# Ensure 'base_path' is defined in a previous cell (e.g., from Ejercicio 2b or 3)
# Example: base_path = "/content/drive/MyDrive/Colab Notebooks/MNA/nlp-mna21-aj-2026-A01796976/Semana08/"
with open(base_path + "palabras_clave_lda.json", "r", encoding="utf-8") as f:
    palabras_clave = json.load(f)

print(f"✅ Cargadas {len(palabras_clave)} fábulas\n")

# --- Generar ---
# This prompt fulfills both Exercise 5a (single summary) and 5b (three subtopics)
def generar_prompt(palabras):
    return f"""Palabras clave: {', '.join(palabras)}

Escribe EXACTAMENTE con este formato:

RESUMEN: [un único enunciado que resume la fábula]

SUBTEMAS:
- [primer subtema]
- [segundo subtema]
- [tercer subtema]
"""

print("-- Generando contenido (Ejercicio 5a y 5b) ---")
resultados = {}

for i, (nombre, palabras) in enumerate(palabras_clave.items(), 1):
    print(f"  [{i}/{len(palabras_clave)}] {nombre}")
    # Handle potential errors during content generation
    try:
        respuesta = modelo.generate_content(generar_prompt(palabras))
        resultados[nombre] = respuesta.text
    except Exception as e:
        print(f"  ❌ Error al generar contenido para {nombre}: {e}")
        resultados[nombre] = f"Error: {e}" # Store error message
    time.sleep(5) # Add a 5-second pause to avoid quota limits

# --- Guardar ---
with open(base_path + "resultados_llm.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=4)

print("\n💾 Guardado: resultados_llm.json")

# --- Mostrar todos los resultados ---
print("\n" + "=" * 60)
print("TODOS LOS RESULTADOS DEL LLM")
print("=" * 60)
if resultados:
    for nombre_fabula, contenido in resultados.items():
        print(f"\n📖 {nombre_fabula}")
        print(contenido)
else:
    print("No se generaron resultados para mostrar.")

✅ API Key obtenida
✅ Cargadas 10 fábulas

-- Generando contenido (Ejercicio 5a y 5b) ---
  [1/10] 21144-14.mp3


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1192.77ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1304.32ms


  [2/10] 21144-04.mp3
  [3/10] 21144-05.mp3
  [4/10] 21144-26.mp3


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5723.18ms


  ❌ Error al generar contenido para 21144-26.mp3: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 24.638830188s.
  [5/10] 21144-06.mp3


  ❌ Error al generar contenido para 21144-06.mp3: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 19.354135683s.
  [6/10] 21144-25.mp3
  [7/10] 21144-27.mp3
  [8/10] 21144-22.mp3


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2561.07ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 991.09ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 941.52ms


  [9/10] 21144-01.mp3


  ❌ Error al generar contenido para 21144-01.mp3: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 20.988605898s.
  [10/10] 21144-24.mp3

💾 Guardado: resultados_llm.json

TODOS LOS RESULTADOS DEL LLM

📖 21144-14.mp3
RESUMEN: Un cabrito, con serenidad y seguridad, enfrenta y replica a un lobo que lo insulta, demostrando valor al saberse protegido y confiado en la seguridad de su sitio.

SUBTEMAS:
-   La demostración de valor y serenidad ante la agresión.
-   La importancia de la seguridad y la protección, ya sea física o

# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**



None

# **Fin de la actividad LDA y LMM: audio-a-texto**

## Conclusiones Finales

En esta actividad, hemos explorado un pipeline completo de procesamiento de lenguaje natural (NLP) desde la extracción de audio hasta la generación de resúmenes y subtemas utilizando un Modelo de Lenguaje Grande (LLM). Los pasos clave incluyen:

1.  **Extracción de Audio a Texto (Whisper):** Utilizamos el modelo Whisper para transcribir fábulas de Esopo en español desde archivos de audio. Este proceso demostró ser efectivo para convertir el habla en texto, aunque siempre es importante considerar la calidad del audio y la claridad del habla para obtener los mejores resultados.

2.  **Limpieza de Texto Personalizada:** Desarrollamos y refinamos una función `eliminar_inicio_final_comun` utilizando expresiones regulares para remover frases introductorias y conclusivas repetitivas, así como créditos de LibriVox, Amara.org y otros sitios web. Esta limpieza fue crucial para aislar el contenido principal de cada fábula, asegurando que el análisis posterior se centrara únicamente en la narrativa relevante.

3.  **Procesamiento de Texto y Eliminación Mínima (NLTK):** Aplicamos una limpieza adicional que incluyó la conversión a minúsculas, la eliminación de caracteres no alfabéticos y la remoción de stopwords. Justificamos la decisión de aplicar una limpieza *mínima* dada la naturaleza concisa de las fábulas, para no perder información semántica vital que podría afectar la capacidad del LDA y del LLM para comprender el contexto.

4.  **Extracción de Palabras Clave (LDA):** Empleamos Latent Dirichlet Allocation (LDA) para identificar las palabras clave más representativas de cada fábula. Este método nos permitió condensar el contenido de las fábulas en un conjunto manejable de términos, que luego servirían como entrada para el LLM. La selección de un solo tópico por fábula con `n_components=1` fue adecuada para este propósito específico de resumir el tema principal.

5.  **Generación de Resúmenes y Subtemas (Gemini LLM):** Finalmente, utilizamos el API de Gemini para generar, a partir de las palabras clave obtenidas, un resumen conciso y tres subtemas para cada fábula. Este paso demostró el poder de los LLMs para comprender y sintetizar información textual compleja, transformando listas de palabras clave en narrativas estructuradas y significativas. La gestión de cuotas y la implementación de pausas (`time.sleep`) fueron aspectos prácticos importantes al interactuar con servicios de LLM para evitar interrupciones.

En general, este ejercicio ha proporcionado una comprensión profunda de cómo combinar diferentes técnicas de NLP, desde el preprocesamiento de datos hasta la generación de lenguaje, para abordar un problema práctico de extracción y resumen de información.